# 1- Marge

Rappelons que la marge d’un point donné est définie comme suit :
$$\gamma(x, y, \theta, \theta_0) = \frac{y(\theta \cdot x + \theta_0)}{\|\theta\|}$$

Le problème : Un séparateur peut avoir une marge immense pour $99\%$ des points, mais passer à $0{,}0001$ millimètre d'un point critique (ou même mal le classer). Évaluer un séparateur sur un seul point ne donne aucune garantie sur sa qualité globale.

Par conséquent, nous souhaitons trouver une fonction de score $S$ pour un séparateur $(\theta, \theta_0)$, de telle manière que la maximisation de *S* permette d’obtenir un meilleur séparateur.

### 1. Le critere de la somme des marge (Margin Inovera)
Marge Inovera (dérivé de « Margin over all » (la marge sur l'ensemble des données)) suggère que, puisque de grandes marge bénéficiaires sont souhaitables, nous devrions maximiser la somme de toutes ces marge bénéficiaires. Elle définit donc cela comme suit :
$$S_{\text{sum}}(\theta, \theta_0) = \sum_{i} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

### 2. Le critère de la Marge Minimale (Minnie Malle)

Pour éviter que des points très éloignés ne masquent de mauvais classements, on définit le score du séparateur par sa **marge minimale** (le pire cas) :

$$S_{\text{min}}(\theta, \theta_0) = \min_{i} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

* **Objectif :** Maximiser $S_{\text{min}}$ revient à éloigner la frontière le plus possible des points les plus proches des deux classes.
* **Résultat :** C'est le principe fondateur des **SVM (Support Vector Machines)** et des classifieurs à marge maximale.

### 3. Le critère de la Marge Maximale (Maxim Argent)

Maxim Argent propose de définir le score par la **marge maximale** :

$$S_{\text{max}}(\theta, \theta_0) = \max_{i} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

* **Problème :** C'est un anti-exemple. Ce score ignore totalement les erreurs commises sur l'ensemble du jeu de données tant qu'au moins un point est très éloigné du bon côté.

Examinons les données suivantes, ainsi que les deux séparateurs possibles (le rouge et le bleu).

```python

data = np.array([[1, 2, 1, 2, 10, 10.3, 10.5, 10.7],
                 [1, 1, 2, 2,  2,  2,  2, 2]])
labels = np.array([[-1, -1, 1, 1, 1, 1, 1, 1]])
blue_th = np.array([[0, 1]]).T
blue_th0 = -1.5
red_th = np.array([[1, 0]]).T
red_th0 = -2.5


```

In [1]:
import numpy as np

In [2]:
data = np.array([[1, 2, 1, 2, 10, 10.3, 10.5, 10.7], [1, 1, 2, 2,  2,  2,  2, 2]])

labels = np.array([[-1, -1, 1, 1, 1, 1, 1, 1]])

blue_th = np.array([[0, 1]]).T
blue_th0 = -1.5

red_th = np.array([[1, 0]]).T
red_th0 = -2.5


In [3]:
data.shape

(2, 8)

In [4]:
red_th.shape

(2, 1)

Quelles sont les valeurs de chaque score $(S_{\text{sum}}, S_{\text{min}}, S_{{\text{max}}})$

In [5]:
# Ssum
s_sum_blue = np.sum((labels * ((blue_th.T@data) + blue_th0)) / np.linalg.norm(blue_th))
s_sum_red  = np.sum((labels * ((red_th.T@data) + red_th0)) / np.linalg.norm(red_th))

# Smin
s_min_blue = np.min((labels * ((blue_th.T@data) + blue_th0)) / np.linalg.norm(blue_th))
s_min_red  = np.min((labels * ((red_th.T@data) + red_th0)) / np.linalg.norm(red_th))

# Smax
s_max_blue = np.max((labels * ((blue_th.T@data) + blue_th0)) / np.linalg.norm(blue_th))
s_max_red  = np.max((labels * ((red_th.T@data) + red_th0)) / np.linalg.norm(red_th))

print(f"Ssum blue = {s_sum_blue}")
print(f"Smin blue = {s_min_blue}")
print(f"Smax blue = {s_max_blue}")

print(f"Ssum red = {s_sum_red}")
print(f"Smin red = {s_min_red}")
print(f"Smax blue = {s_max_red}")

print(f"Blue: [{s_sum_blue}, {s_min_blue}, {s_max_blue}]")
print(f"Red: [{s_sum_red}, {s_min_red}, {s_max_red}]")


Ssum blue = 4.0
Smin blue = 0.5
Smax blue = 0.5
Ssum red = 31.5
Smin red = -1.5
Smax blue = 8.2
Blue: [4.0, 0.5, 0.5]
Red: [31.5, -1.5, 8.2]


![1A](./assets/q1A.png)

```python
[31.5, -1.5, 8.2]
```

![1BQ](./assets/q1B.png)

```python
[4.0, 0.5, 0.5]
```

![1CQ](./assets/q1C.png)

red

![1DQ](./assets/q1D.png)

blue

![1EQ](./assets/q1E.png)

red

![1FQ](./assets/q1F.png)

$S_{\text{min}}$

# 2. Perte (Loss)

Sur la base de ce qui a été dit précédemment, nous avons décidé d’essayer de trouver un séparateur linéaire $(\theta, \theta_0)$ qui maximise la marge minimale (c’est-à-dire la distance entre le séparateur et les points qui lui sont les plus proches). Nous définissons la marge d’un ensemble de données $(X, Y)$ par rapport à un séparateur comme étant :

$$\gamma(X, Y, \theta, \theta_0) = \min_{i=1, \dots, n} \frac{y^{(i)}(\theta^T x^{(i)} + \theta_0)}{\|\theta\|}$$

ou sous forme condensée :

$$\gamma(X, Y, \theta, \theta_0) = \min_{i=1, \dots, n} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

une façon de résoudre ce problème consiste à définir une valeur $\gamma_{\text{ref}}$ pour la marge du jeu de données, puis à chercher un séparateur linéaire qui maximise la valeur $\gamma_{\text{ref}}$

![2A](./assets/2AR.png)

![2B](./assets/2B.png)

![2C](./assets/2C.png)

Nous voulons maintenant améliorer cette marge de garantie, qui est extrêmement faible, dans l’algorithme de Perceptron. Comme nous l’avons vu lors du cours, une manière efficace de concevoir des algorithmes d’apprentissage consiste à les présenter comme des problèmes d’optimisation, puis à utiliser des stratégies d’optimisation de nature générale pour les résoudre.

Une forme typique du problème d’optimisation consiste à minimiser un objectif dont la forme est la suivante :

$$J(\theta, \theta_0) = \frac{1}{n} \sum_{i=1}^{n} L(x^{(i)}, y^{(i)}, \theta, \theta_0) + \lambda R(\theta, \theta_0)$$

![note](./assets/note.png)

![2D](./assets/2D.png)

![2E](./assets/2E.png)

# 3. Simply inseparable 

Nous préférerions une fonction de perte qui aide à orienter le processus d’optimisation vers une solution adéquate, surtout dans le cas où les données sont linéairement séparables. De plus, dans les ensembles de données réels, il est relativement rare que les données soient linéairement séparables. Par conséquent, notre algorithme doit être capable de gérer ce cas également, tout en cherchant à trouver un séparateur linéaire optimal, même s’il n’est pas parfait. Au lieu d’utiliser la fonction de perte $(0, \infty)$ , nous devrions concevoir une fonction de perte qui nous permette de relâcher la contrainte selon laquelle tous les points doivent respecter un certain écart minimal par rapport au séparateur linéaire, tout en continuant à favoriser des écarts importants entre les points.

![loss](./assets/image.png)

![3A](./assets/3A.png)

```python
data = np.array([[1.1, 1, 4],[3.1, 1, 2]])
labels = np.array([[1, -1, -1]])
th = np.array([[1, 1]]).T
th0 = -4
```

![3B](./assets/3B.png)

data = np.array([[1.1, 1, 4],[3.1, 1, 2]])
labels = np.array([[1, -1, -1]])
th = np.array([[1, 1]]).T
th0 = -4

In [6]:
y_ref = np.sqrt(2) / 2

In [8]:
data = np.array([[1.1, 1, 4],[3.1, 1, 2]])
labels = np.array([[1, -1, -1]])
th = np.array([[1, 1]]).T
th0 = -4

In [9]:
margin = (labels * ((th.T@data) + th0)) / np.linalg.norm(th)
lh = np.where(y_ref > margin, 1 - (margin/y_ref), 0)

In [10]:
lh

array([[0.8, 0. , 3. ]])

In [11]:
import sympy as sp

In [12]:
exact = np.vectorize(sp.nsimplify)(lh)

print(exact)

[[4/5 0 3]]


![3B1](./assets/3B1.png)

# 4) It hinges on the loss


![formule](./assets/4_formule.png)

![4_A](./assets/4_A.png)

![4_B](./assets/4_B.png)

![4_C](./assets/4_C.png)

# 5) Linear Support Vector Machines

### 5) Machines à Vecteurs de Support Linéaires (SVM)

L'objectif d'entraînement d'une Machine à Vecteurs de Support (avec marge douce / *slack*) peut être vu comme la recherche d'un équilibre entre la perte Hinge moyenne sur les exemples et un terme de régularisation qui cherche à garder $\theta$ petit (ce qui revient à maximiser la marge). Cet équilibre est contrôlé par le paramètre de régularisation $\lambda$.

Ici, nous considérons uniquement le cas sans le paramètre de biais $\theta_0$ (en le fixant à zéro) et nous réécrivons l'objectif d'entraînement sous la forme d'une moyenne :

$$\left[ \frac{1}{n} \sum_{i=1}^n L_h\left(y^{(i)} \theta \cdot x^{(i)}\right) \right] + \frac{\lambda}{2} \|\theta\|^2 = \frac{1}{n} \sum_{i=1}^n \left[ L_h\left(y^{(i)} \theta \cdot x^{(i)}\right) + \frac{\lambda}{2} \|\theta\|^2 \right]$$

où $L_h\left(y(\theta \cdot x)\right) = \max\{0, 1 - y(\theta \cdot x)\}$ est la perte Hinge (remarque : nous écrirons aussi parfois la perte Hinge sous la forme $L_h(v) = \max(0, 1 - v)$).

Nous pouvons désormais minimiser cette fonction objectif globale à l'aide de l'algorithme **Pegasos**, qui choisit de manière itérative un exemple d'entraînement au hasard et applique une règle de mise à jour par descente de gradient basée sur le terme correspondant entre crochets du côté droit.

Dans ce problème, nous allons optimiser l'objectif d'entraînement en utilisant un **seul exemple d'entraînement**, afin de mieux comprendre comment le paramètre de régularisation $\lambda$ influence le résultat. À cette fin, nous désignons cet unique exemple d'entraînement par la paire (vecteur de caractéristiques, étiquette) $(x, y)$. Nous chercherons ensuite à trouver un $\theta$ qui minimise :

$$J_\lambda^{(1)}(\theta) \equiv L_h(y(\theta \cdot x)) + \frac{\lambda}{2} \|\theta\|^2$$

Dans les sous-parties suivantes, nous allons montrer que le $\theta$ minimisant $J_\lambda^{(1)}$, noté $\hat{\theta}$, est nécessairement de la forme :

$$\hat{\theta} = \eta y x$$

pour un certain réel $\eta > 0$.

![5_A](./assets/5_A.png)

![5_B](./assets/5_B.png)

![5_C](./assets/5_C.png)

### 5D)

![5_D](./assets/5_D.png)

![5_E](./assets/5_E.png)

# 6) Implementing gradient descent

### 6.1) Gradient descent

> **Note:** If you need a refresher on gradient descent, you may want to reference this week's notes.

We want to find the $x$ that minimizes the value of the objective function $f(x)$, for an arbitrary scalar function $f$. The function $f$ will be implemented as a Python function of one argument, which will be a NumPy column vector. For efficiency, we will work with Python functions that return not just the value of $f(x)$, but also return the gradient vector at $x$, that is, $\nabla_x f(x)$.

We will now implement a generic gradient descent function, `gd`, that has the following input arguments:

* **`f`**: a function whose input is an $x$ (a column vector) and returns a scalar.
* **`df`**: a function whose input is an $x$ (a column vector) and returns a column vector representing the gradient of $f$ at $x$.
* **`x0`**: an initial value of $x$, $x_0$, which is a column vector.
* **`step_size_fn`**: a function that is given the iteration index (an integer) and returns a step size.
* **`max_iter`**: the number of iterations to perform.

Our function `gd` returns a tuple `(x, fs, xs)`:
* **`x`**: the value of $x$ at the final step.
* **`fs`**: the list of values of $f(x)$ found during all the iterations (including $f(x_0)$).
* **`xs`**: the list of values of $x$ found during all the iterations (including $x_0$).

**Hints:**
* **Hint 1:** This is a short function!
* **Hint 2:** If you do `temp_x = x` where `x` is a vector (NumPy array), then `temp_x` is just another name for the same vector as `x`, and changing an entry in one will change an entry in the other. You should either use `x.copy()` or remember to change entries back after modification.

*(Some test or example functions that you may find useful are included below. You may also find `rv` and `cv` from previous weeks useful, though not necessary).*

In [57]:
def rv(value_list):
    return np.array([value_list])

def cv(value_list):
    return np.transpose(rv(value_list))

def f1(x):
    return ((2 * x + 3)**2).item()

def df1(x):
    return 2 * 2 * (2 * x + 3)

def f2(v):
    x = float(v[0, 0]); y = float(v[1, 0])
    return (x - 2.) * (x - 3.) * (x + 3.) * (x + 1.) + (x + y -1)**2

def df2(v):
    x = float(v[0, 0]); y = float(v[1, 0])
    return cv([(-3. + x) * (-2. + x) * (1. + x) + \
               (-3. + x) * (-2. + x) * (3. + x) + \
               (-3. + x) * (1. + x) * (3. + x) + \
               (-2. + x) * (1. + x) * (3. + x) + \
               2 * (-1. + x + y),
               2 * (-1. + x + y)])

In [58]:
def gd(f, df, x0, step_size_fn, max_iter):
    x = x0.copy()
    
    fs = [f(x)]
    xs = [x.copy()]
    
    for i in range(max_iter):
        x = x - step_size_fn(i) * df(x)
        
        fs.append(f(x))
        xs.append(x.copy())
        
    return (x, fs, xs)

In [59]:
def package_ans(gd_vals):
    x, fs, xs = gd_vals
    return [x.tolist(), [fs[0], fs[-1]], [xs[0].tolist(), xs[-1].tolist()]]

In [61]:
# Test case 1
ans=package_ans(gd(f1, df1, cv([0.]), lambda i: 0.1, 1000))
print(f"Result 1 -- {ans}")

# Test case 2
ans=package_ans(gd(f2, df2, cv([0., 0.]), lambda i: 0.01, 1000))
print(f"Result 2 -- {ans}")

Result 1 -- [[[-1.5]], [9.0, 0.0], [[[0.0]], [[-1.5]]]]
Result 2 -- [[[-2.2058239041648853], [3.205823890926977]], [19.0, -20.967239611348752], [[[0.0], [0.0]], [[-2.2058239041648853], [3.205823890926977]]]]


![6_1](./assets/6_1.png)

In [69]:
def num_grad(f, delta=0.001):
    def df(x):
        d = len(x)
        grad = np.zeros((d, 1))
        
        for i in range(d):
            x_plus = x.copy()
            x_minus = x.copy()
            
            x_plus[i, 0] += delta
            x_minus[i, 0] -= delta
            
            f_plus = f(x_plus)
            f_minus = f(x_minus)
                
            grad[i, 0] = (f_plus - f_minus) / (2 * delta)
            
        return grad

    return df

In [70]:
x = cv([0.])
ans=(num_grad(f1)(x).tolist(), x.tolist())
print(f"Result 1 -- {ans}")

x = cv([0.1])
ans=(num_grad(f1)(x).tolist(), x.tolist())
print(f"Result 2 -- {ans}")

x = cv([0., 0.])
ans=(num_grad(f2)(x).tolist(), x.tolist())
print(f"Result 3 -- {ans}")

x = cv([0.1, -0.1])
ans=(num_grad(f2)(x).tolist(), x.tolist())
print(f"Result 4 -- {ans}")

Result 1 -- ([[11.999999999998678]], [[0.0]])
Result 2 -- ([[12.799999999999478]], [[0.1]])
Result 3 -- ([[6.99999899999959], [-2.000000000000668]], [[0.0], [0.0]])
Result 4 -- ([[4.7739994000011166], [-2.000000000000668]], [[0.1], [-0.1]])


![6_2](./assets/6_2.png)

In [71]:
def minimize(f, x0, step_size_fn, max_iter):
    df = num_grad(f, delta=0.001)
    return gd(f, df, x0, step_size_fn, max_iter)

In [72]:
ans = package_ans(minimize(f1, cv([0.]), lambda i: 0.1, 1000))
print(f"Result 1 -- {ans}")


ans = package_ans(minimize(f2, cv([0., 0.]), lambda i: 0.01, 1000))
print(f"Result 2 -- {ans}")

Result 1 -- [[[-1.5]], [9.0, 0.0], [[[0.0]], [[-1.5]]]]
Result 2 -- [[[-2.2058237062057517], [3.205823692967833]], [19.0, -20.967239611347775], [[[0.0], [0.0]], [[-2.2058237062057517], [3.205823692967833]]]]


![6_3](./assets/6_3.png)

##### Note: Donc en gros, dans la pratique, pour avoir le gradient, on se base sur l'estimation de la derivee qu'on a vu en DIC1 Semestre 2 dans le module de Calculs Numeriques dans le chapitre 2 intitule differenciation numerique (Difference finie centree): Cours: https://drive.google.com/file/d/13659aykNcIi5Jij5rZ0W4Z7Mg3Ep7cVl/view?usp=sharing
##### Cette formule se base uniquement sur la fonction elle meme on fait des evaluations a gauche et a droite...

<div align='center'>
    <img src='./assets/cours1.png' alt='cours1' style="border-radius: 15px;" />
    <br/><br/>
    <img src='./assets/cours2.png' alt='cours2' style="border-radius: 15px;" />
</div>

# Approximation de $f'(x)$ par Différences Finies Centrées

## 1. Développements de Taylor autour de $x$

Par la formule de Taylor avec un pas $h > 0$ :

$$f(x+h) = f(x) + h f'(x) + \frac{h^2}{2!} f''(x) + \frac{h^3}{3!} f^{(3)}(x) + \frac{h^4}{4!} f^{(4)}(x) + \frac{h^5}{5!} f^{(5)}(x) + \mathcal{O}(h^6) \quad (1)$$

En remplaçant $h$ par $-h$, on obtient :

$$f(x-h) = f(x) - h f'(x) + \frac{h^2}{2!} f''(x) - \frac{h^3}{3!} f^{(3)}(x) + \frac{h^4}{4!} f^{(4)}(x) - \frac{h^5}{5!} f^{(5)}(x) + \mathcal{O}(h^6) \quad (2)$$

---

## 2. Soustraction des développements

Calculons la différence $(1) - (2)$ terme à terme :

* **Terme $f(x)$ :** $f(x) - f(x) = 0$ *(s'annule)*
* **Terme $f'(x)$ :** $h f'(x) - (-h f'(x)) = 2h f'(x)$ *(subsiste)*
* **Terme $f''(x)$ :** $\frac{h^2}{2!} f''(x) - \frac{h^2}{2!} f''(x) = 0$ *(s'annule)*
* **Terme $f^{(3)}(x)$ :** $\frac{h^3}{3!} f^{(3)}(x) - \left(-\frac{h^3}{3!} f^{(3)}(x)\right) = \frac{2h^3}{3!} f^{(3)}(x)$ *(subsiste)*
* **Terme $f^{(4)}(x)$ :** $\frac{h^4}{4!} f^{(4)}(x) - \frac{h^4}{4!} f^{(4)}(x) = 0$ *(s'annule)*
* **Terme $f^{(5)}(x)$ :** $\frac{h^5}{5!} f^{(5)}(x) - \left(-\frac{h^5}{5!} f^{(5)}(x)\right) = \frac{2h^5}{5!} f^{(5)}(x)$ *(subsiste)*

On obtient ainsi :

$$f(x+h) - f(x-h) = 2h f'(x) + \frac{2h^3}{3!} f^{(3)}(x) + \frac{2h^5}{5!} f^{(5)}(x) + \mathcal{O}(h^7)$$

---

## 3. Isolation de $f'(x)$ et Approximation

En divisant la relation par $2h$ :

$$\frac{f(x+h) - f(x-h)}{2h} = f'(x) + \frac{h^2}{6} f^{(3)}(x) + \frac{h^4}{120} f^{(5)}(x) + \mathcal{O}(h^6)$$

Soit en isolant la dérivée première :

$$f'(x) = \frac{f(x+h) - f(x-h)}{2h} - \frac{h^2}{6} f^{(3)}(x) - \mathcal{O}(h^4)$$

Pour $h$ suffisamment petit ($h \ll 1$), en négligeant les termes d'ordre supérieur en $h^2$, la **formule des différences finies centrées** s'écrit :

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

---

> **Analyse de l'Erreur**
>
> * **Annulation des ordres pairs :** Les puissances paires de $h$ vérifient $(-h)^{2k} = h^{2k}$. La soustraction $(1) - (2)$ élimine donc automatiquement tous les termes d'ordre pair ($f'', f^{(4)}, \dots$).
> * **Ordre de grandeur :** L'erreur de troncature est de l'ordre de **$\mathcal{O}(h^2)$**.
> * **Avantage :** Ce schéma est plus précis que la *différence progressive* $\frac{f(x+h) - f(x)}{h}$ dont l'erreur est seulement en **$\mathcal{O}(h)$**.

# 7.1) Calculating the SVM objective

In [85]:
def hinge(v):
    return np.where((v < 1), 1 - v, 0)

# x is dxn, y is 1xn, th is dx1, th0 is 1x1
def hinge_loss(x, y, th, th0):
    margin = y * ((x.T @ th + th0).T)
    n = x.shape[1]
    return np.sum(hinge(margin)) / n

# x is dxn, y is 1xn, th is dx1, th0 is 1x1, lam is a scalar
def svm_obj(x, y, th, th0, lam):
    return hinge_loss(x, y, th, th0) + (lam*(np.linalg.norm(th)**2))

In [86]:
def super_simple_separable():
    X = np.array([[2, 3, 9, 12],
                  [5, 2, 6, 5]])
    y = np.array([[1, -1, 1, -1]])
    return X, y

sep_e_separator = np.array([[-0.40338351], [1.1849563]]), np.array([[-2.26910091]])

# Test case 1
x_1, y_1 = super_simple_separable()
th1, th1_0 = sep_e_separator
ans = svm_obj(x_1, y_1, th1, th1_0, .1)
print(f"Result 1 -- {ans}")

# Test case 2
ans = svm_obj(x_1, y_1, th1, th1_0, 0.0)
print(f"Result 2 -- {ans}")

Result 1 -- 0.15668396890496103
Result 2 -- 0.0


<div align='center'>
    <img src='./assets/7_1.png' alt='7_1' style="border-radius: 15px;" />
</div>

### 7.2) Calculating the SVM gradient

In [ ]:
# Returns the gradient of hinge(v) with respect to v.
def d_hinge(v):
    return None

# Returns the gradient of hinge_loss(x, y, th, th0) with respect to th
def d_hinge_loss_th(x, y, th, th0):
    return None

# Returns the gradient of hinge_loss(x, y, th, th0) with respect to th0
def d_hinge_loss_th0(x, y, th, th0):
    return None

# Returns the gradient of svm_obj(x, y, th, th0) with respect to th
def d_svm_obj_th(x, y, th, th0, lam):
    return None

# Returns the gradient of svm_obj(x, y, th, th0) with respect to th0
def d_svm_obj_th0(x, y, th, th0, lam):
    return None

# Returns the full gradient as a single vector (which includes both th, th0)
def svm_obj_grad(X, y, th, th0, lam):
    return None


## 7.2) Calculating the SVM gradient

In [88]:
# Returns the gradient of hinge(v) with respect to v.
def d_hinge(v):
    return np.where(v < 1, -1, 0)

# Returns the gradient of hinge_loss(x, y, th, th0) with respect to th
def d_hinge_loss_th(x, y, th, th0):
    n = x.shape[1]
    margin = y * ((x.T @ th + th0).T)
    d_loss = d_hinge(margin)
    return (1/n) * np.sum(y * (d_loss * x), axis=1, keepdims=True)

# Returns the gradient of hinge_loss(x, y, th, th0) with respect to th0
def d_hinge_loss_th0(x, y, th, th0):
    n = x.shape[1]
    margin = y * ((x.T @ th + th0).T)
    d_loss = d_hinge(margin)
    return (1/n) * np.sum(y * (d_loss), axis=1, keepdims=True)

# Returns the gradient of svm_obj(x, y, th, th0) with respect to th
def d_svm_obj_th(x, y, th, th0, lam):
    return d_hinge_loss_th(x, y, th, th0) + (2 * lam * th)

# Returns the gradient of svm_obj(x, y, th, th0) with respect to th0
def d_svm_obj_th0(x, y, th, th0, lam):
    return d_hinge_loss_th0(x, y, th, th0)

# Returns the full gradient as a single vector (which includes both th, th0)
def svm_obj_grad(X, y, th, th0, lam):
    return np.vstack((d_svm_obj_th(X, y, th, th0, lam), d_svm_obj_th0(X, y, th, th0, lam)))


In [89]:
X1 = np.array([[1, 2, 3, 9, 10]])
y1 = np.array([[1, 1, 1, -1, -1]])
th1, th10 = np.array([[-0.31202807]]), np.array([[1.834     ]])
X2 = np.array([[2, 3, 9, 12],
               [5, 2, 6, 5]])
y2 = np.array([[1, -1, 1, -1]])
th2, th20=np.array([[ -3.,  15.]]).T, np.array([[ 2.]])

d_hinge(np.array([[ 71.]])).tolist()
d_hinge(np.array([[ -23.]])).tolist()
d_hinge(np.array([[ 71, -23.]])).tolist()

d_hinge_loss_th(X2[:,0:1], y2[:,0:1], th2, th20).tolist()
d_hinge_loss_th(X2, y2, th2, th20).tolist()
d_hinge_loss_th0(X2[:,0:1], y2[:,0:1], th2, th20).tolist()
d_hinge_loss_th0(X2, y2, th2, th20).tolist()

d_svm_obj_th(X2[:,0:1], y2[:,0:1], th2, th20, 0.01).tolist()
d_svm_obj_th(X2, y2, th2, th20, 0.01).tolist()
d_svm_obj_th0(X2[:,0:1], y2[:,0:1], th2, th20, 0.01).tolist()
d_svm_obj_th0(X2, y2, th2, th20, 0.01).tolist()

svm_obj_grad(X2, y2, th2, th20, 0.01).tolist()
svm_obj_grad(X2[:,0:1], y2[:,0:1], th2, th20, 0.01).tolist()

[[-0.06], [0.3], [0.0]]

<div align='center'>
    <img src='./assets/7_2-1.png' alt='7_1' style="border-radius: 15px;" />
    <img src='./assets/7_2-2.png' alt='7_1' style="border-radius: 15px;" />
</div>

## 7.3) Batch SVM minimize

In [93]:
def batch_svm_min(data, labels, lam):
   def svm_min_step_size_fn(i):
      return 2/(i+1)**0.5

   init_data = np.zeros(shape=(data.shape[0] + 1, 1))

   def f(W):
      th = W[:-1]
      th0 = W[-1:]
      return svm_obj(data, labels, th, th0, lam)

   def df(W):
      th = W[:-1]
      th0 = W[-1:]
      return svm_obj_grad(data, labels, th, th0, lam)

   return gd(f, df, init_data, svm_min_step_size_fn, 10)


In [94]:
def separable_medium():
    X = np.array([[2, -1, 1, 1],
                  [-2, 2, 2, -1]])
    y = np.array([[1, -1, 1, -1]])
    return X, y
sep_m_separator = np.array([[ 2.69231855], [ 0.67624906]]), np.array([[-3.02402521]])

x_1, y_1 = super_simple_separable()
ans = package_ans(batch_svm_min(x_1, y_1, 0.0001))
print(f"Result 1 -- {ans}")

x_1, y_1 = separable_medium()
ans = package_ans(batch_svm_min(x_1, y_1, 0.0001))
print(f"Result 2 -- {ans}")

Result 1 -- [[[-1.4810507930100065], [4.406219189763341], [-0.40377305279098563]], [np.float64(1.0), np.float64(2.4572174098028015)], [[[0.0], [0.0], [0.0]], [[-1.4810507930100065], [4.406219189763341], [-0.40377305279098563]]]]
Result 2 -- [[[1.4460693132604328], [0.7975607987934759], [-1.2082511097181943]], [np.float64(1.0), np.float64(0.33814559335191335)], [[[0.0], [0.0], [0.0]], [[1.4460693132604328], [0.7975607987934759], [-1.2082511097181943]]]]


<div align='center'>
    <img src='./assets/7_3.png' alt='7_3' style="border-radius: 15px;" />
</div>

7.4) Numerical SVM objective (Optional)
Recall from the previous question that we were able to closely approximate gradients with numerical estimates. We may apply the same technique to optimize the SVM objective.

Using your definition of minimize and num_grad from the previous problem, implement a function that optimizes the SVM objective through numeric approximations.

How well does this function perform, compared to the analytical result? Consider both accuracy and runtime.